# Synthetic UNO Scene Generator

Inspired by [geaxgx/playing-card-detection](https://github.com/geaxgx/playing-card-detection).

Pipeline:
1. Load each card image + its two corner hulls from `annotations.csv` (HL = upper-left symbol, LR = lower-right symbol).
2. Random rotation + perspective warp on each card. Track where the hulls land.
3. Compose scenes on a random background:
   - **no-overlap**: 1–6 cards, rejected if their warped quads intersect.
   - **occlusion**: 2–3 cards stacked geaxgx-style. A card is kept only if at least one of its two corner hulls remains fully uncovered by the cards placed after it.
4. For each visible hull, write a YOLO bbox (the axis-aligned box around the warped hull).

In [ ]:
from __future__ import annotations
import os, glob, random, math, json
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from shapely.geometry import Polygon
from shapely.affinity import affine_transform
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
CARDS_DIR = DATA_DIR / 'card_images'
BG_DIR = DATA_DIR / 'backgrounds'
OUT_DIR = ROOT / 'synthetic'
OUT_IMG = OUT_DIR / 'images'
OUT_LBL = OUT_DIR / 'labels'
for d in (OUT_IMG, OUT_LBL):
    d.mkdir(parents=True, exist_ok=True)

# Output scene canvas size — 3:2 aspect to match the real train images (4000x2662).
SCENE_W, SCENE_H = 1080, 720
# Card resize: pick a target longest-side so cards look natural on the canvas
CARD_LONG_SIDE = 220

SEED = 42
random.seed(SEED); np.random.seed(SEED)

print('Cards:', len(list(CARDS_DIR.glob('*.jpg'))))
print('Backgrounds:', len(list(BG_DIR.glob('*'))))

In [ ]:
# --- Load card sprites + corner hulls -------------------------------------
ann = pd.read_csv(DATA_DIR / 'annotations.csv')

CLASS_NAMES = sorted({p.stem for p in CARDS_DIR.glob('*.jpg')})
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
print(f'{len(CLASS_NAMES)} classes')

def load_card_bgra(path: Path) -> np.ndarray:
    """Load a JPG card and add a full-opaque alpha channel."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    alpha = np.full(img.shape[:2], 255, dtype=np.uint8)
    return np.dstack([img, alpha])

def hulls_for(card_filename: str):
    """Return list of (label, polygon_pts Nx2) for each corner of a card."""
    rows = ann[ann['image'] == card_filename]
    polys = []
    for _, r in rows.iterrows():
        x, y, w, h = int(r['posx']), int(r['posy']), int(r['width']), int(r['height'])
        # Rectangular hull around the corner symbol
        pts = np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], dtype=np.float32)
        polys.append((r['corner'], pts))
    return polys

CARDS = []  # list of dicts: {name, img_bgra, hulls:[Nx2], size:(w,h)}
for jpg in sorted(CARDS_DIR.glob('*.jpg')):
    img = load_card_bgra(jpg)
    h, w = img.shape[:2]
    polys = hulls_for(jpg.name)
    if not polys:
        # No annotation -> fallback: skip warning, still include card with empty hulls
        print(f'warn: no hulls for {jpg.name}')
    CARDS.append({'name': jpg.stem, 'file': jpg.name, 'img': img,
                  'hulls': polys, 'size': (w, h)})
print(f'Loaded {len(CARDS)} cards. Example hulls for {CARDS[0]["name"]}:',
      [(c, p.tolist()) for c, p in CARDS[0]['hulls']])


In [ ]:
# --- Card transform: scale, rotate, perspective ---------------------------

def transform_points(M: np.ndarray, pts: np.ndarray) -> np.ndarray:
    """Apply 3x3 homography (or 2x3 affine) to Nx2 points."""
    pts = np.asarray(pts, dtype=np.float32)
    if M.shape == (2, 3):
        ones = np.ones((pts.shape[0], 1), dtype=np.float32)
        return (np.hstack([pts, ones]) @ M.T)
    ones = np.ones((pts.shape[0], 1), dtype=np.float32)
    homog = np.hstack([pts, ones]) @ M.T
    return homog[:, :2] / homog[:, 2:3]


def warp_card(card: dict,
              scale: float = 1.0,
              angle: float | None = None,
              perspective: float = 0.10):
    """
    Returns:
        warped_bgra: HxWx4 image of the warped card on a transparent canvas
        warped_hulls: list of (label, Nx2 polygon in canvas coords)
        card_quad: 4x2 polygon of the warped card outline (canvas coords)
    """
    src = card['img']
    h, w = src.shape[:2]

    # 1. Resize to target long-side
    long_side = max(h, w)
    s = (CARD_LONG_SIDE / long_side) * scale
    new_w, new_h = int(w * s), int(h * s)
    resized = cv2.resize(src, (new_w, new_h), interpolation=cv2.INTER_AREA)
    scale_M = np.array([[s, 0, 0], [0, s, 0]], dtype=np.float32)
    hulls_s = [(lbl, transform_points(scale_M, pts)) for lbl, pts in card['hulls']]

    # 2. Rotation
    if angle is None:
        angle = random.uniform(-180, 180)
    cx, cy = new_w / 2, new_h / 2
    R = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    # Compute bounding canvas after rotation
    cos, sin = abs(R[0, 0]), abs(R[0, 1])
    rot_w = int(new_h * sin + new_w * cos)
    rot_h = int(new_h * cos + new_w * sin)
    R[0, 2] += rot_w / 2 - cx
    R[1, 2] += rot_h / 2 - cy
    rotated = cv2.warpAffine(resized, R, (rot_w, rot_h),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    hulls_r = [(lbl, transform_points(R, pts)) for lbl, pts in hulls_s]
    card_corners = np.array([[0, 0], [new_w, 0], [new_w, new_h], [0, new_h]], dtype=np.float32)
    quad = transform_points(R, card_corners)

    # 3. Mild perspective warp (random push of each corner)
    H, W = rotated.shape[:2]
    src_pts = np.array([[0, 0], [W, 0], [W, H], [0, H]], dtype=np.float32)
    jitter = perspective * min(W, H)
    dst_pts = src_pts + np.random.uniform(-jitter, jitter, src_pts.shape).astype(np.float32)
    # Re-frame so dst is in positive coords
    minxy = dst_pts.min(axis=0)
    dst_pts -= minxy
    out_w = int(math.ceil(dst_pts[:, 0].max())) + 1
    out_h = int(math.ceil(dst_pts[:, 1].max())) + 1
    Hp = cv2.getPerspectiveTransform(src_pts, dst_pts)
    persp = cv2.warpPerspective(rotated, Hp, (out_w, out_h),
                                flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    hulls_p = [(lbl, transform_points(Hp, pts)) for lbl, pts in hulls_r]
    quad_p = transform_points(Hp, quad)
    return persp, hulls_p, quad_p


def paste_bgra(canvas: np.ndarray, sprite: np.ndarray, x: int, y: int) -> np.ndarray:
    """Alpha-blend `sprite` onto `canvas` at top-left (x, y). Returns canvas mutated."""
    H, W = canvas.shape[:2]
    sh, sw = sprite.shape[:2]
    x0, y0 = max(x, 0), max(y, 0)
    x1, y1 = min(x + sw, W), min(y + sh, H)
    if x1 <= x0 or y1 <= y0:
        return canvas
    sx0, sy0 = x0 - x, y0 - y
    sx1, sy1 = sx0 + (x1 - x0), sy0 + (y1 - y0)
    region = canvas[y0:y1, x0:x1]
    patch = sprite[sy0:sy1, sx0:sx1]
    alpha = patch[..., 3:4].astype(np.float32) / 255.0
    region[..., :3] = (patch[..., :3].astype(np.float32) * alpha
                      + region[..., :3].astype(np.float32) * (1 - alpha)).astype(np.uint8)
    if canvas.shape[2] == 4:
        region[..., 3] = np.maximum(region[..., 3], patch[..., 3])
    canvas[y0:y1, x0:x1] = region
    return canvas


In [ ]:
# --- Background loader & scene composers ----------------------------------

BG_FILES = sorted([p for p in BG_DIR.glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])

def random_background() -> np.ndarray:
    """Random crop of a random background image, resized to (SCENE_H, SCENE_W)."""
    if not BG_FILES:
        # Fallback: solid noise
        return np.random.randint(0, 255, (SCENE_H, SCENE_W, 3), dtype=np.uint8)
    bg = cv2.imread(str(random.choice(BG_FILES)))
    h, w = bg.shape[:2]
    # Random crop a square then resize
    side = min(h, w)
    y0 = random.randint(0, h - side)
    x0 = random.randint(0, w - side)
    bg = bg[y0:y0 + side, x0:x0 + side]
    return cv2.resize(bg, (SCENE_W, SCENE_H))


def place_card_on_scene(scene_bgr: np.ndarray, card: dict,
                        cx: int, cy: int,
                        scale: float = 1.0,
                        angle: float | None = None):
    """Warp card, paste it centered at (cx, cy) on scene. Returns (hulls_scene, quad_scene)."""
    sprite, hulls, quad = warp_card(card, scale=scale, angle=angle)
    sh, sw = sprite.shape[:2]
    x = cx - sw // 2
    y = cy - sh // 2
    # Translate hulls/quad into scene coords
    hulls_scene = [(lbl, pts + np.array([x, y], dtype=np.float32)) for lbl, pts in hulls]
    quad_scene = quad + np.array([x, y], dtype=np.float32)
    # Convert scene to BGRA temporarily for alpha paste
    bgra = np.dstack([scene_bgr, np.full(scene_bgr.shape[:2], 255, dtype=np.uint8)])
    paste_bgra(bgra, sprite, x, y)
    scene_bgr[:] = bgra[..., :3]
    return hulls_scene, quad_scene


def quad_to_poly(quad: np.ndarray) -> Polygon:
    return Polygon(quad).buffer(0)


def compose_no_overlap(n_cards: int, max_tries: int = 30):
    """Place n cards on a random background, rejecting any whose quad intersects existing ones."""
    scene = random_background()
    placed = []  # list of dicts: {card, hulls, quad}
    placed_polys = []
    chosen = random.sample(CARDS, k=min(n_cards, len(CARDS)))
    for card in chosen:
        for _ in range(max_tries):
            sprite, hulls, quad = warp_card(card,
                                            scale=random.uniform(0.7, 1.1))
            sh, sw = sprite.shape[:2]
            if sw >= SCENE_W or sh >= SCENE_H:
                continue
            x = random.randint(0, SCENE_W - sw)
            y = random.randint(0, SCENE_H - sh)
            hulls_s = [(lbl, pts + np.array([x, y], dtype=np.float32)) for lbl, pts in hulls]
            quad_s = quad + np.array([x, y], dtype=np.float32)
            poly = quad_to_poly(quad_s)
            if not poly.is_valid or poly.area < 100:
                continue
            if any(poly.intersects(p) for p in placed_polys):
                continue
            # Accept
            bgra = np.dstack([scene, np.full(scene.shape[:2], 255, dtype=np.uint8)])
            paste_bgra(bgra, sprite, x, y)
            scene = bgra[..., :3].copy()
            placed.append({'card': card, 'hulls': hulls_s, 'quad': quad_s})
            placed_polys.append(poly)
            break
    return scene, placed


def compose_occlusion(n_cards: int = 3, max_tries: int = 30):
    """
    geaxgx-style: stack n cards. For each previously-placed card, after pasting the
    new one, require at least one of its corner hulls to remain un-occluded.
    Cards whose all hulls are covered are dropped from the scene labels.
    """
    assert 2 <= n_cards <= 4
    scene = random_background()
    chosen = random.sample(CARDS, k=n_cards)
    placed = []  # {card, hulls (orig scene coords), quad}
    # Pick a shared center region so cards actually overlap
    cx0 = random.randint(SCENE_W // 3, 2 * SCENE_W // 3)
    cy0 = random.randint(SCENE_H // 3, 2 * SCENE_H // 3)
    for card in chosen:
        success = False
        for _ in range(max_tries):
            sprite, hulls, quad = warp_card(card, scale=random.uniform(0.8, 1.05))
            sh, sw = sprite.shape[:2]
            jitter = 90
            cx = cx0 + random.randint(-jitter, jitter)
            cy = cy0 + random.randint(-jitter, jitter)
            x = cx - sw // 2
            y = cy - sh // 2
            if x < 0 or y < 0 or x + sw > SCENE_W or y + sh > SCENE_H:
                continue
            hulls_s = [(lbl, pts + np.array([x, y], dtype=np.float32)) for lbl, pts in hulls]
            quad_s = quad + np.array([x, y], dtype=np.float32)
            new_poly = quad_to_poly(quad_s)
            if not new_poly.is_valid:
                continue
            # Check previously-placed cards: each must still have at least one hull
            # not (significantly) covered by this new card.
            ok = True
            for prev in placed:
                visible = False
                for lbl, pts in prev['hulls']:
                    hp = quad_to_poly(pts)
                    if not hp.intersects(new_poly):
                        visible = True
                        break
                    inter = hp.intersection(new_poly).area
                    if inter / max(hp.area, 1e-6) < 0.05:  # <5% covered = still readable
                        visible = True
                        break
                if not visible:
                    ok = False
                    break
            if not ok:
                continue
            # Accept and paste
            bgra = np.dstack([scene, np.full(scene.shape[:2], 255, dtype=np.uint8)])
            paste_bgra(bgra, sprite, x, y)
            scene = bgra[..., :3].copy()
            placed.append({'card': card, 'hulls': hulls_s, 'quad': quad_s})
            success = True
            break
        if not success:
            # Could not place; stop adding cards rather than retry forever
            break

    # Post-pass: recompute hull visibility against all *later*-placed cards
    final = []
    for i, p in enumerate(placed):
        visible_hulls = []
        for lbl, pts in p['hulls']:
            hp = quad_to_poly(pts)
            covered = False
            for q in placed[i + 1:]:
                later = quad_to_poly(q['quad'])
                if later.intersects(hp) and later.intersection(hp).area / max(hp.area, 1e-6) > 0.05:
                    covered = True
                    break
            if not covered:
                visible_hulls.append((lbl, pts))
        if visible_hulls:
            final.append({'card': p['card'], 'hulls': visible_hulls, 'quad': p['quad']})
    return scene, final


In [ ]:
# --- YOLO label conversion + dataset writer -------------------------------

def yolo_lines_from_placed(placed, img_w=SCENE_W, img_h=SCENE_H):
    """For each visible hull, emit one YOLO line: 'cls cx cy w h' (normalized)."""
    lines = []
    for p in placed:
        cls_id = CLASS_TO_ID[p['card']['name']]
        for lbl, pts in p['hulls']:
            xs, ys = pts[:, 0], pts[:, 1]
            x0, x1 = max(xs.min(), 0), min(xs.max(), img_w - 1)
            y0, y1 = max(ys.min(), 0), min(ys.max(), img_h - 1)
            if x1 <= x0 or y1 <= y0:
                continue
            cx, cy = (x0 + x1) / 2 / img_w, (y0 + y1) / 2 / img_h
            w, h = (x1 - x0) / img_w, (y1 - y0) / img_h
            lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines


def save_sample(stem: str, scene_bgr: np.ndarray, placed):
    img_path = OUT_IMG / f'{stem}.jpg'
    lbl_path = OUT_LBL / f'{stem}.txt'
    cv2.imwrite(str(img_path), scene_bgr)
    lbl_path.write_text('\n'.join(yolo_lines_from_placed(placed)))


def write_classes_file():
    (OUT_DIR / 'classes.txt').write_text('\n'.join(CLASS_NAMES))
    yaml = (
        f"path: {OUT_DIR}\n"
        "train: images\n"
        "val: images\n"
        f"nc: {len(CLASS_NAMES)}\n"
        "names: [" + ", ".join(f"'{n}'" for n in CLASS_NAMES) + "]\n"
    )
    (OUT_DIR / 'data.yaml').write_text(yaml)


def generate_dataset(n_no_overlap: int = 500,
                      n_occlusion: int = 500,
                      cards_no_overlap=(1, 6),
                      cards_occlusion=(2, 3)):
    write_classes_file()
    for i in tqdm(range(n_no_overlap), desc='no-overlap'):
        n = random.randint(*cards_no_overlap)
        scene, placed = compose_no_overlap(n)
        if placed:
            save_sample(f'no_{i:05d}', scene, placed)
    for i in tqdm(range(n_occlusion), desc='occlusion'):
        n = random.randint(*cards_occlusion)
        scene, placed = compose_occlusion(n)
        if placed:
            save_sample(f'occ_{i:05d}', scene, placed)
    print('Dataset at', OUT_DIR)


In [ ]:
# --- Preview a few samples ------------------------------------------------

def draw_preview(scene_bgr, placed):
    out = scene_bgr.copy()
    for p in placed:
        cv2.polylines(out, [p['quad'].astype(np.int32)], True, (0, 255, 0), 2)
        for lbl, pts in p['hulls']:
            cv2.polylines(out, [pts.astype(np.int32)], True, (0, 0, 255), 2)
            x0, y0 = pts[:, 0].min(), pts[:, 1].min()
            x1, y1 = pts[:, 0].max(), pts[:, 1].max()
            cv2.rectangle(out, (int(x0), int(y0)), (int(x1), int(y1)), (255, 255, 0), 1)
            cv2.putText(out, p['card']['name'], (int(x0), int(y0) - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    return out


fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    if i < 3:
        scene, placed = compose_no_overlap(random.randint(2, 5))
        ax.set_title(f'no-overlap ({len(placed)} cards)')
    else:
        scene, placed = compose_occlusion(3)
        ax.set_title(f'occlusion ({len(placed)} visible)')
    ax.imshow(cv2.cvtColor(draw_preview(scene, placed), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# --- Run full generation --------------------------------------------------
# Tune the counts below; ~1000 total is a reasonable starter set.
generate_dataset(n_no_overlap=300, n_occlusion=300)
